In [24]:
from collections import Counter

In [25]:
#I want to check if user_bank_payment is a valid subset of bank, if it is remove the necessary payment from the bank list
#first i want to do this without a function and build from there

#Now add logic to pay with properties if you can

money_charged = 4

bank  = [5,3]
property_value = [2,2,3,3]

user_bank_payment = [3] #subset of bank
user_property_payment = [2]

#############################

bank_counts = Counter(bank)
user_payment_counts = Counter(user_bank_payment)

total_payment = sum(user_bank_payment)
if total_payment < money_charged:
    print("Payment is less than required amount.")
else:
    for user_card, user_count in user_payment_counts.items():
        if user_count > bank_counts.get(user_card, 0):
            print(f"Payment includes {user_card} which is not in bank or exceeds available count.")
            break
        elif user_count <= bank_counts.get(user_card, 0):
            #remove the necessary payment from the bank list
            bank.remove(user_card)
    print("Payment Processed.")


Payment is less than required amount.


## Final Payment Processing Code

In [26]:
#Payment processing code v2

#Split the payment from the removing from bank and property list logic

def remove_payment_from_list(payment_list, payment_counts):
    for payment_card, payment_count in payment_counts.items():
        for _ in range(payment_count):
            payment_list.remove(payment_card)


def process_payment(money_charged,
                    bank,
                    properties,
                    user_bank_payment,
                    user_property_payment):
    
    bank_counts = Counter(bank)
    properties_counts = Counter(properties)
    user_bank_payment_counts = Counter(user_bank_payment)
    user_property_payment_counts = Counter(user_property_payment)

    available_total = sum(bank) + sum(properties)
    offered_total = sum(user_bank_payment) + sum(user_property_payment)

    #Validate user is paying with cards they have
    for payment_card, payment_count in user_bank_payment_counts.items():
        if payment_count > bank_counts.get(payment_card, 0):
            print(f"Payment includes {payment_card} which is not in bank or exceeds available count.")
            raise ValueError("Invalid bank payment.")
        
    for payment_card, payment_count in user_property_payment_counts.items():
        if payment_count > properties_counts.get(payment_card, 0):
            print(f"Payment includes {payment_card} which is not in properties or exceeds available count.")
            raise ValueError("Invalid property payment.")
        
    paid_all_bank = (user_bank_payment_counts == bank_counts)
    paid_all_props = (user_property_payment_counts == properties_counts)

    if offered_total >= money_charged:
        print("Valid Payment")
        remove_payment_from_list(bank, user_bank_payment_counts)
        remove_payment_from_list(properties, user_property_payment_counts)
    elif (available_total < money_charged) and (offered_total == available_total) and paid_all_bank and paid_all_props:
        print("Valid Payment with all available cards")
        remove_payment_from_list(bank, user_bank_payment_counts)
        remove_payment_from_list(properties, user_property_payment_counts)
    else:
        print("Payment is less than required amount.")
        raise ValueError("Insufficient payment.")
    
    return(bank,properties)


In [27]:
#Testing the functions

money_charged = 21

bank  = [5,4,3]
properties = [2,2]

user_bank_payment = [5,4,3]
user_property_payment = [2,2]

bank, properties = process_payment(money_charged,
                bank,
                properties,
                user_bank_payment,
                user_property_payment)

bank

Valid Payment with all available cards


[]

## Structure of Application

## JSON card structure

In [28]:
import os, json

base_path = os.getcwd()

all_cards_json_path = os.path.join(base_path, "all_cards.json")

all_cards_json = json.load(open(all_cards_json_path, "r"))
cards_path = os.path.join(base_path,"backend" ,"cards", "base")

for card in all_cards_json:
    card_filename = f"{card['name']}.json"
    card_path = os.path.join(cards_path, card_filename)
    with open(card_path, "w") as f:
        json.dump(card, f, indent=4)


### Pydantic Models:

JSON -> Validate with Raw Pydantic Models -> CardDef for engine gameplay

In [29]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Any, Dict, List, Optional, Literal

class RawEffect(BaseModel):
    type: str
    model_config = ConfigDict(extra="allow")  # keep other keys

class RawCard(BaseModel):
    id: str
    name: str
    type: str                 # your JSON uses "type"
    count: int
    bank_value: int

    colors: Optional[List[str]] = None
    property_group: Optional[str] = None
    effect: Optional[RawEffect] = None
    rent_by_count: Optional[List[int]] = None

    bankable: Optional[bool] = None
    image_url: Optional[str] = None
    restrictions: Optional[Dict[str, Any]] = None
    note: Optional[str] = None

    model_config = ConfigDict(extra="allow")  # keep any future fields

class PlayDef(BaseModel):
    effect: str
    params: Dict[str, Any] = Field(default_factory=dict)

class CardDef(BaseModel):
    id: str
    name: str
    kind: Literal["money","property","property_wild","rent","action"]
    copies: int
    money_value: int
    colors: Optional[List[str]] = None
    rent_by_count: Optional[List[int]] = None 
    play: Optional[PlayDef] = None
    meta: Dict[str, Any] = Field(default_factory=dict)



In [30]:
#Convert RawCard to CardDef
def raw_to_carddef(raw: RawCard) -> Optional[CardDef]:
    kind_map = {
        "money": "money",
        "property": "property",
        "property_wild": "property_wild",
        "rent": "rent",
        "action": "action"
    }
    
    kind = kind_map.get(raw.type)
    if not kind:
        return None  # Unknown type
    
    play_def = None
    if raw.effect:
        play_def = PlayDef(effect=raw.effect.type, params={k: v for k, v in raw.effect.model_dump().items() if k != "type"})
    
    card_def = CardDef(
        id=raw.id,
        name=raw.name,
        kind=kind,
        copies=raw.count,
        money_value=raw.bank_value,
        rent_by_count=raw.rent_by_count,
        colors=raw.colors,
        play=play_def,
        meta={
            "property_group": raw.property_group,
            "bankable": raw.bankable,
            "image_url": raw.image_url,
            "restrictions": raw.restrictions,
            "note": raw.note
        }
    )
    
    return card_def

In [31]:
#Load a single card definition from JSON file


def load_card_defs(path: str) -> list[CardDef]:
    data = json.loads(open(path, "r", encoding="utf-8").read())
    raw_cards = RawCard.model_validate(data)
    #raw_cards = [RawCard.model_validate(x) for x in data]
    #defs = [raw_to_carddef(rc) for rc in raw_cards]
    return raw_cards


load_card_defs(os.path.join(cards_path, "Illinois Avenue.json"))

RawCard(id='prop_red_illinois_avenue', name='Illinois Avenue', type='property', count=1, bank_value=3, colors=None, property_group='red', effect=None, rent_by_count=[2, 3, 6], bankable=True, image_url='https://monopolydealrules.com/photos/cards/red-property-card.png', restrictions=None, note=None, set_size=3)

#### Build the Main Card Catalog Function

In [32]:
#Load all cards

from pathlib import Path
import json
from typing import Dict, List

def load_card_defs_from_dir(cards_dir: str) -> Dict[str, CardDef]:
    catalog: Dict[str,CardDef] = {}

    for path in Path(cards_dir).glob("*.json"):
        raw_data = json.loads(path.read_text(encoding="utf-8"))

        # Normalize to a list
        if isinstance(raw_data, list):
            raw_cards = [RawCard.model_validate(item) for item in raw_data]
        else:
            raw_cards = [RawCard.model_validate(raw_data)]

        for rc in raw_cards:
            cd = raw_to_carddef(rc)
            if cd is None:
                continue
            if cd.id in catalog:
                raise ValueError(f"Duplicate card ID found: {cd.id}")
            catalog[cd.id] = cd

    return catalog



### Build rent table from catalog (This will be done once per game)

In [46]:

def build_rent_table(catalog: Dict[str, CardDef]) -> Dict[str, List[int]]:
    rent_table: Dict[str, List[int]] = {}
    for card_def in catalog.values():
        if card_def.kind == "property" and card_def.meta.get("property_group") and card_def.rent_by_count:
            # All properties in a group should have same rent_by_count
            rent_table[card_def.meta["property_group"]] = card_def.rent_by_count
    return rent_table

#Example on how to use!
catalog = load_card_defs_from_dir(cards_path)

RENT_TABLE = build_rent_table(catalog)
RENT_TABLE

{'yellow': [2, 4, 6],
 'railroad': [1, 2, 3, 4],
 'brown': [1, 2],
 'dark_blue': [3, 8],
 'light_blue': [1, 2, 3],
 'utility': [1, 2],
 'red': [2, 3, 6],
 'orange': [1, 3, 5],
 'green': [2, 4, 7],
 'pink': [1, 2, 4]}

## GameState, PlayerState, and DeckState

In [ ]:
from typing import Dict, List, Optional
from pydantic import BaseModel, Field


class DeckState(BaseModel):
    draw_pile: List[str] = Field(default_factory = list)
    discard_pile: List[str] = Field(default_factory = list)

class PlayerState(BaseModel):
    id: str
    hand: List[str] = Field(default_factory=list)        # card ids
    bank: List[str] = Field(default_factory=list)        # card ids
    properties: Dict[str, List[str]] = Field(default_factory=dict)  # color -> card ids
    buildings: Dict[str, List[str]] = Field(default_factory=dict)  # color -> building card ids

class GameState(BaseModel):
    id: str
    players: Dict[str, PlayerState]
    deck: DeckState
    current_player_id: Optional[str] = None
    turn_number: int = 1
    actions_taken: int = 0




## Create a Deck using Pydantic Models we built, Create a new game, and Draw Cards functions


### Build Deck function

In [35]:
import random
from typing import Dict, List

def build_deck(catalog: Dict[str, CardDef], seed = 42) -> List[str]:
    deck: List[str] = []

    #catalog is key: card id, value: CardDef
    for cd in catalog.values():
        deck.extend([cd.id] * cd.copies)
    
    #Shuffle deck with seed
    random.seed(seed)
    random.shuffle(deck)

    #Check if deck is 110 cards long

    if len(deck) != 106: #110 - 4 helper cards
        
        raise ValueError(f"Deck size is {len(deck)}, expected 110.")
    
    return deck


#Test this function out

catalog = load_card_defs_from_dir(cards_path)
deck = build_deck(catalog, seed=123)



### Create a new game function

In [36]:
import uuid
from typing import Dict, List
#from .engine.state import GameState, PlayerState, DeckState #TODO: we will add this later
#from .services.card_catalog import build_deck #TODO: we will also add this later

STARTING_HAND_SIZE = 5

def create_new_game(player_ids: List[str], 
                    catalog: Dict[str, CardDef]) -> GameState:
    # 1) Build and shuffle deck
    draw_pile = build_deck(catalog)
    deck = DeckState(draw_pile=draw_pile, discard_pile=[])

    # 2) Create players
    players = {pid: PlayerState(id=pid) for pid in player_ids}

    # 3) Deal starting hands
    for _ in range(STARTING_HAND_SIZE):
        for pid in player_ids:
            if not deck.draw_pile:
                raise ValueError("Deck is empty while dealing starting hands.")
            card_id = deck.draw_pile.pop(0)
            players[pid].hand.append(card_id)

    # 4) Create game state
    return GameState(
        id=str(uuid.uuid4()),
        players=players,
        deck=deck,
        current_player_id=player_ids[0] if player_ids else None,
        turn_number=1,
    )

### Draw Card Function

In [37]:
import random
from typing import List
#from .state import GameState #TODO: Add this file structure later

def draw_cards(state: GameState, player_id: str, n: int = 1) -> GameState:
    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    for _ in range(n):
        if not state.deck.draw_pile:
            if not state.deck.discard_pile:
                raise ValueError("Cannot draw: no cards left in draw or discard pile.")
            # move discard into draw and shuffle
            state.deck.draw_pile = state.deck.discard_pile
            state.deck.discard_pile = []
            random.shuffle(state.deck.draw_pile)

        card_id = state.deck.draw_pile.pop(0)
        state.players[player_id].hand.append(card_id)

    return state

### Discard Function

In [38]:
from typing import Iterable
#from .state import GameState #TODO: Add this file structure later

def discard_cards(state: GameState, 
                  player_id: str, 
                  card_ids: Iterable[str]) -> GameState:
    
    """"
    Card_ids function argument is the card ids to be discarded from the player's hand that the player chooses to discard.
    We take this directly from the UI and input it into the engine to update the game state.
    """
    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    hand = state.players[player_id].hand
    for cid in card_ids:
        if cid not in hand:
            raise ValueError(f"Player {player_id} does not have card {cid} in hand.")
        hand.remove(cid)
        state.deck.discard_pile.append(cid)

    return state


### End Turn Function

In [39]:
from typing import List
#from .state import GameState #TODO: Add this file structure later

def end_turn(state: GameState, turn_order: List[str]) -> GameState:

    if not turn_order:
        raise ValueError("turn_order is empty.")
    if state.current_player_id not in turn_order:
        raise ValueError("current_player_id not found in turn_order.")

    idx = turn_order.index(state.current_player_id)
    next_idx = (idx + 1) % len(turn_order)

    state.current_player_id = turn_order[next_idx]
    state.turn_number += 1
    state.actions_taken = 0
    
    return state

### Win Check

In [40]:
#TODO: Check to see if you can make this more efficient by keeping track of complete sets as the game progresses!
from typing import Dict, Optional
#from .state import GameState, PlayerState #TODO: Add this file structure later

def count_complete_sets(player: PlayerState, set_sizes: Dict[str, int]) -> int:
    complete = 0
    for color, cards in player.properties.items():
        needed = set_sizes.get(color)
        if needed is not None and len(cards) >= needed:
            complete += 1
    return complete

def check_win(
    state: GameState,
    property_set_sizes: Dict[str, int],
) -> Optional[str]:
    for player_id, player in state.players.items():
        if count_complete_sets(player, property_set_sizes) >= 3:
            return player_id
    return None


## Playing Functions 

**This is the main gameplay functions that will be used in the game. Later we can add character passives and special abilities that will make the game more interesting to play**


In [41]:
def use_action(state: GameState, max_actions: int = 3) -> None:
    if state.actions_taken >= max_actions:
        raise ValueError("No actions remaining this turn.")
    state.actions_taken += 1

### Play Property

In [42]:
from typing import Dict, Optional
# from .state import GameState
# from ..schemas.card_defs import CardDef  # adjust path if needed #TODO: Add this file structure later

def play_property(state: GameState,
                  catalog: Dict[str, CardDef],
                  player_id: str,
                  card_id: str,
                  color_if_wild: Optional[str] = None) -> GameState:
    #Increment actions taken by 1 (change if there are exceptions later)
    use_action(state)

    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    player = state.players[player_id]

    if card_id not in player.hand:
        raise ValueError("Card not in player hand.")

    if card_id not in catalog:
        raise ValueError(f"Unknown card id: {card_id}")

    cd = catalog[card_id]

    if cd.kind not in {"property", "property_wild"}:
        raise ValueError("Card is not a property or a wild property.")

    if not cd.colors:
        raise ValueError("Property card has no colors defined.")

    # Choose color
    if cd.kind == "property":
        # If only one color, auto-use it unless a different color was passed

        if len(cd.colors) == 1:
            chosen_color = cd.colors[0]
            if color_if_wild and color_if_wild != chosen_color:
                raise ValueError(f"Invalid color '{color_if_wild}' for this property.")
        else:
            # Multi-color property behaves like wild; require explicit choice
            if not color_if_wild or color_if_wild not in cd.colors:
                raise ValueError("Must choose a valid color for this property.")
            chosen_color = color_if_wild
    else:
        # property_wild: must choose one of its colors
        if not color_if_wild or color_if_wild not in cd.colors:
            raise ValueError("Must choose a valid color for a wild property.")
        chosen_color = color_if_wild

    # Move card to properties
    player.hand.remove(card_id)
    player.properties.setdefault(chosen_color, []).append(card_id)

    return state
    
    


### Play Bank function

In [43]:
from typing import Dict
# from .state import GameState
# from ..schemas.card_defs import CardDef  # adjust import path if needed #TODO: Add this file structure later

def play_bank(
    state: GameState,
    player_id: str,
    card_id: str,
    catalog: Dict[str, CardDef],
) -> GameState:
    
    #Increment actions taken by 1 (change if there are exceptions later)
    use_action(state)

    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    player = state.players[player_id]

    if card_id not in player.hand:
        raise ValueError("Card not in player hand.")

    if card_id not in catalog:
        raise ValueError(f"Unknown card id: {card_id}")

    cd = catalog[card_id]

    # In Monopoly Deal, any card can be banked for its money value
    if cd.money_value <= 0 and cd.kind == "property": #Can't bank property cards since while they have money value, they are properties
        raise ValueError("This card has no money value or it's a property and cannot be banked.") #For example this can be a wild multiproperty card

    # Move from hand -> bank
    player.hand.remove(card_id)
    player.bank.append(card_id)

    return state


### Charge rent function

In [ ]:
#Add extra rent if you have a building in a appropriate full property set
#Can only have houses and then hotels on a full property set
#No houses or hotels on railroads or utilities

def building_bonus_for_color(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    color: str,
) -> int:
    bonus = 0
    for bid in state.players[player_id].buildings.get(color, []):
        cd = catalog[bid]
        if cd.play and cd.play.effect == "building":
            bonus += int(cd.play.params.get("rent_bonus", 0))
    return bonus

In [47]:
from collections import Counter
from typing import Dict, List, Optional

# Expect this to exist once you load cards:
# RENT_TABLE = build_rent_table(raw_cards_or_catalog)

def charge_rent_amount(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    rent_card_id: str,
    color: str,
    double_rent_ids: Optional[List[str]] = None,
) -> int:
    
    if "RENT_TABLE" not in globals():
        raise ValueError("RENT_TABLE is not defined. Build it once before charging rent.")
    rent_table = RENT_TABLE

    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")
    player = state.players[player_id]

    # Ensure player has the rent card + any double rent cards
    hand_counts = Counter(player.hand)
    play_ids = [rent_card_id] + (double_rent_ids or [])
    play_counts = Counter(play_ids)
    for cid, cnt in play_counts.items():
        if cnt > hand_counts.get(cid, 0):
            raise ValueError(f"Player does not have {cnt} copies of {cid} in hand.")

    # Validate rent card
    if rent_card_id not in catalog:
        raise ValueError(f"Unknown card id: {rent_card_id}")
    rent_card = catalog[rent_card_id]
    if rent_card.kind != "rent":
        raise ValueError("Card is not a rent card.")

    # Validate color choice
    allowed_colors = rent_card.colors or []
    if "any" not in allowed_colors and color not in allowed_colors:
        raise ValueError(f"Rent card cannot be used for color '{color}'.")

    # Base rent from RENT_TABLE
    if color not in rent_table:
        raise ValueError(f"No rent table defined for color '{color}'.")
    rent_steps = rent_table[color]
    if not rent_steps:
        raise ValueError(f"Rent table for '{color}' is empty.")

    prop_count = len(player.properties.get(color, []))
    if prop_count == 0:
        raise ValueError(f"Player has no properties in color '{color}'.")
    step_index = min(prop_count, len(rent_steps)) - 1
    base_rent = rent_steps[step_index]

    #Apply Building bonus only on full sets
    # only allow building bonus if full set
    full_set = prop_count >= len(rent_steps)
    bonus = building_bonus_for_color(state, catalog, player_id, color) if full_set else 0


    # Apply Double The Rent modifiers (stackable)
    multiplier = 1
    for dr_id in (double_rent_ids or []):
        if dr_id not in catalog:
            raise ValueError(f"Unknown card id: {dr_id}")
        dr = catalog[dr_id]

        if dr.kind != "action" or not dr.play:
            raise ValueError(f"{dr_id} is not a valid Double The Rent card.")
        
        if dr.play.effect != "modifier" or dr.play.params.get("applies_to") != "rent":
            raise ValueError(f"{dr_id} is not a valid rent modifier.")
        multiplier *= int(dr.play.params.get("multiplier", 2))

    return (base_rent+bonus) * multiplier


### Play actions

In [ ]:
from typing import Dict, List, Optional, Any

def resolve_targets(
    state: GameState,
    actor_id: str,
    target_spec: str,
    target_player_id: Optional[str],
) -> List[str]:
    
    if target_spec == "all_others":
        return [pid for pid in state.players.keys() if pid != actor_id]
    if target_spec == "one_player":
        if not target_player_id:
            raise ValueError("target_player_id is required.")
        if target_player_id not in state.players:
            raise ValueError(f"Unknown target_player_id: {target_player_id}")
        if target_player_id == actor_id:
            raise ValueError("Cannot target yourself.")
        return [target_player_id]
    if target_spec == "self":
        return [actor_id]
    raise ValueError(f"Unknown target spec: {target_spec}")

def discard_action_cards(state: GameState, actor: PlayerState, card_ids: List[str]) -> None:
    for cid in card_ids:
        if cid not in actor.hand:
            raise ValueError(f"Card {cid} not in hand.")
        actor.hand.remove(cid)
        state.deck.discard_pile.append(cid)

def play_action(
    state: GameState,
    catalog: Dict[str, CardDef],
    actor_id: str,
    action_card_id: str,
    *,
    # rent params
    rent_color: Optional[str] = None,
    double_rent_ids: Optional[List[str]] = None,
    # targeting
    target_player_id: Optional[str] = None,
    # property manipulation params
    steal_card_id: Optional[str] = None,
    give_card_id: Optional[str] = None,
    steal_color: Optional[str] = None,
    # payment selections for each target (if you want to process immediately)
    payment_selections: Optional[Dict[str, Dict[str, List[str]]]] = None,

    change_wild: Optional[Dict[str, str]] = None
) -> Dict[str, Any]:
    if actor_id not in state.players:
        raise ValueError(f"Unknown actor_id: {actor_id}")

    actor = state.players[actor_id]

    if action_card_id not in actor.hand:
        raise ValueError("Action card not in actor hand.")
    if action_card_id not in catalog:
        raise ValueError(f"Unknown card id: {action_card_id}")

    card = catalog[action_card_id]

    # Count 1 action for the whole play
    use_action(state)

    # Rent cards are type "rent"
    if card.kind == "rent":
        if not card.play or card.play.effect != "collect_rent":
            raise ValueError("Rent card missing collect_rent effect.")

        if not rent_color:
            raise ValueError("rent_color is required for rent cards.")

        target_spec = card.play.params.get("target")
        targets = resolve_targets(state, actor_id, target_spec, target_player_id)

        amount = charge_rent_amount(
            state=state,
            catalog=catalog,
            player_id=actor_id,
            rent_card_id=action_card_id,
            color=rent_color,
            double_rent_ids=double_rent_ids,
        )

        # Discard rent card + any double-rent cards here (Option A)
        discard_action_cards(state, actor, [action_card_id] + (double_rent_ids or []))

        # If payment selections provided, process immediately
        receipts = []
        if payment_selections:
            for tgt in targets:
                sel = payment_selections.get(tgt)
                if not sel:
                    raise ValueError(f"No payment selection provided for target {tgt}")
                receipts.append(
                    process_payment(
                        state=state,
                        payer_id=tgt,
                        catalog=catalog,
                        user_bank_payment_ids=sel.get("bank", []),
                        user_property_payment_ids=sel.get("properties", []),
                        money_charged=amount,
                    )
                )

        return {
            "type": "rent",
            "amount": amount,
            "targets": targets,
            "receipts": receipts,
        }

    # Action cards
    if card.kind != "action":
        raise ValueError("Card is not an action or rent card.")

    if not card.play:
        raise ValueError("Action card missing play info.")

    effect = card.play.effect
    params = card.play.params

    # Draw cards (Pass Go)
    if effect == "draw_cards":
        amount = int(params.get("amount", 0))
        if amount <= 0:
            raise ValueError("draw_cards missing amount.")
        discard_action_cards(state, actor, [action_card_id])
        draw_cards(state, actor_id, amount)
        return {"type": "draw_cards", "amount": amount}

    # Property manipulation (Deal Breaker / Sly Deal / Forced Deal)
    if effect in {"steal_full_set", "steal_property", "swap_property"}:
        discard_action_cards(state, actor, [action_card_id])
        return process_property_manipulation(
            state=state,
            catalog=catalog,
            actor_id=actor_id,
            action_card_id=action_card_id,
            target_player_id=target_player_id,
            steal_card_id=steal_card_id,
            give_card_id=give_card_id,
            steal_color=steal_color,
        )

    # Charge players (Birthday / Debt Collector)
    if effect == "charge_players":
        amount = params.get("amount")
        if amount is None:
            raise ValueError("charge_players missing amount.")
        target_spec = params.get("target")
        targets = resolve_targets(state, actor_id, target_spec, target_player_id)

        discard_action_cards(state, actor, [action_card_id])

        receipts = []
        if payment_selections:
            for tgt in targets:
                sel = payment_selections.get(tgt)
                if not sel:
                    raise ValueError(f"No payment selection provided for target {tgt}")
                receipts.append(
                    process_payment(
                        state=state,
                        payer_id=tgt,
                        catalog=catalog,
                        user_bank_payment_ids=sel.get("bank", []),
                        user_property_payment_ids=sel.get("properties", []),
                        money_charged=int(amount),
                    )
                )

        return {
            "type": "charge_players",
            "amount": int(amount),
            "targets": targets,
            "receipts": receipts,
        }
    
    # Change wild color (no card played, still counts as an action)
    if change_wild:
        card_id = change_wild.get("card_id")
        new_color = change_wild.get("new_color")
        if not card_id or not new_color:
            raise ValueError("change_wild requires card_id and new_color.")
        change_wild_color(
            state=state,
            catalog=catalog,
            player_id=actor_id,
            card_id=card_id,
            new_color=new_color,
        )
        return {"type": "change_wild_color", "card_id": card_id, "new_color": new_color}

    # Just Say No will be handled later
    #TODO: Implement Just Say No Logic Later!!!!
    raise ValueError(f"Unsupported action effect: {effect}")


## Function Handlers/Processers

Charge payment effects

1. It’s My Birthday, Debt Collector, Rent
* Use charge_rent_amount(...) (for rent)
* Use process_payment(...) to actually deduct cards

2. Property manipulation effects

* Deal Breaker, Sly Deal, Forced Deal
* Needs a function that moves property cards between players

3. Draw effects

* Pass Go
* Just call draw_cards(...) for 2 cards

4. So the flow becomes:

    play_action decides what kind of effect the card has
    it calls the appropriate handler:
* process_payment(...)
* process_property_manipulation(...)
* process_draw(...)


### Payment Processing

In [ ]:
from collections import Counter
from typing import Dict, List, Any

def remove_ids_from_list(pool: List[str], ids_to_remove: List[str]) -> None:
    counts = Counter(pool)
    remove_counts = Counter(ids_to_remove)
    for cid, cnt in remove_counts.items():
        if cnt > counts.get(cid, 0):
            raise ValueError(f"Cannot remove {cnt} copies of {cid} from list.")
    for cid in ids_to_remove:
        pool.remove(cid)

def remove_ids_from_properties(properties: Dict[str, List[str]], ids_to_remove: List[str]) -> None:

    to_remove = Counter(ids_to_remove)

    for color, cards in properties.items():
        new_cards = []
        for cid in cards:
            if to_remove[cid] > 0:
                to_remove[cid] -= 1
            else:
                new_cards.append(cid)
        properties[color] = new_cards

    missing = [cid for cid, cnt in to_remove.items() if cnt > 0]
    if missing:
        raise ValueError(f"Card ids not found in properties: {missing}")

def card_value(cid: str, catalog: Dict[str, CardDef]) -> int:
    if cid not in catalog:
        raise ValueError(f"Unknown card id: {cid}")
    return catalog[cid].money_value

def process_payment(
    state: GameState,
    payer_id: str,
    catalog: Dict[str, CardDef],
    user_bank_payment_ids: List[str],
    user_property_payment_ids: List[str],
    money_charged: int,
) -> Dict[str, Any]:
    if payer_id not in state.players:
        raise ValueError(f"Unknown payer_id: {payer_id}")

    payer = state.players[payer_id]

    bank_counts = Counter(payer.bank)
    prop_counts = Counter([cid for cards in payer.properties.values() for cid in cards])

    bank_pay_counts = Counter(user_bank_payment_ids)
    prop_pay_counts = Counter(user_property_payment_ids)

    # Validate player has the offered cards
    for cid, cnt in bank_pay_counts.items():
        if cnt > bank_counts.get(cid, 0):
            raise ValueError(f"Payment includes {cid} not in bank or exceeds count.")

    for cid, cnt in prop_pay_counts.items():
        if cnt > prop_counts.get(cid, 0):
            raise ValueError(f"Payment includes {cid} not in properties or exceeds count.")

    offered_total = sum(card_value(cid, catalog) for cid in user_bank_payment_ids) + \
                    sum(card_value(cid, catalog) for cid in user_property_payment_ids)

    available_total = sum(card_value(cid, catalog) for cid in payer.bank) + \
                      sum(card_value(cid, catalog) for cid in prop_counts.elements())

    paid_all_bank = bank_pay_counts == bank_counts
    paid_all_props = prop_pay_counts == prop_counts

    if offered_total >= money_charged:
        remove_ids_from_list(payer.bank, user_bank_payment_ids)
        remove_ids_from_properties(payer.properties, user_property_payment_ids)
        fully_paid = True
    elif (available_total < money_charged) and (offered_total == available_total) and paid_all_bank and paid_all_props:
        remove_ids_from_list(payer.bank, user_bank_payment_ids)
        remove_ids_from_properties(payer.properties, user_property_payment_ids)
        fully_paid = True
    else:
        raise ValueError("Insufficient payment.")

    return {
        "state": state,
        "payer_id": payer_id,
        "paid_amount": offered_total,
        "bank_cards": user_bank_payment_ids,
        "property_cards": user_property_payment_ids,
        "fully_paid": fully_paid,
    }


### Process Property Manipulation Actions


In [49]:
from typing import Dict, Optional, Any

def get_set_size(color: str) -> int:
    if "RENT_TABLE" not in globals():
        raise ValueError("RENT_TABLE is not defined. Build it once before property actions.")
    if color not in RENT_TABLE:
        raise ValueError(f"No rent table for color '{color}'.")
    return len(RENT_TABLE[color])

def is_full_set(player: PlayerState, color: str) -> bool:
    return len(player.properties.get(color, [])) >= get_set_size(color)

def find_card_color(properties: Dict[str, List[str]], card_id: str) -> str:
    for color, cards in properties.items():
        if card_id in cards:
            return color
    raise ValueError(f"Card id {card_id} not found in properties.")

def process_property_manipulation(
    state: GameState,
    catalog: Dict[str, CardDef],
    actor_id: str,
    action_card_id: str,
    target_player_id: str,
    *,
    steal_card_id: Optional[str] = None,   # Sly Deal & Forced Deal
    give_card_id: Optional[str] = None,    # Forced Deal
    steal_color: Optional[str] = None,     # Deal Breaker (full set)
) -> Dict[str, Any]:
    
    if actor_id not in state.players:
        raise ValueError(f"Unknown actor_id: {actor_id}")
    if target_player_id not in state.players:
        raise ValueError(f"Unknown target_player_id: {target_player_id}")
    if actor_id == target_player_id:
        raise ValueError("Cannot target yourself.")

    actor = state.players[actor_id]
    target = state.players[target_player_id]

    if action_card_id not in catalog:
        raise ValueError(f"Unknown card id: {action_card_id}")

    action_card = catalog[action_card_id]
    if action_card.kind != "action" or not action_card.play:
        raise ValueError("Card is not a playable action.")

    effect = action_card.play.effect
    params = action_card.play.params


    ## Deal Breaker Logic

    if effect == "steal_full_set":
        if not steal_color:
            raise ValueError("steal_color is required for Deal Breaker.")
        if not is_full_set(target, steal_color):
            raise ValueError(f"Target does not have a full set of '{steal_color}'.")

        stolen_cards = target.properties.get(steal_color, [])
        target.properties[steal_color] = [] #The target loses the full set
        actor.properties.setdefault(steal_color, []).extend(stolen_cards) #The actor gains the full set

        #Handle any building transfers if needed:
        if params.get("includes_buildings"):
            if hasattr(target, "buildings") and hasattr(actor, "buildings"):
                b = target.buildings.get(steal_color, [])
                target.buildings[steal_color] = []
                actor.buildings.setdefault(steal_color, []).extend(b)

        return {
            "type": "steal_full_set",
            "actor_id": actor_id,
            "target_id": target_player_id,
            "color": steal_color,
            "cards": stolen_cards,
        }
    ## Sly Deal Logic

    if effect == "steal_property":
        if not steal_card_id:
            raise ValueError("steal_card_id is required for Sly Deal.")
        steal_color_actual = find_card_color(target.properties, steal_card_id)

        if not params.get("from_full_set_allowed", False) and is_full_set(target, steal_color_actual):
            raise ValueError("Cannot steal from a full set.")

        target.properties[steal_color_actual].remove(steal_card_id)
        actor.properties.setdefault(steal_color_actual, []).append(steal_card_id)

        return {
            "type": "steal_property",
            "actor_id": actor_id,
            "target_id": target_player_id,
            "card_id": steal_card_id,
            "color": steal_color_actual,
        }
    ## Forced Deal Logic
    
    if effect == "swap_property":
        if not steal_card_id or not give_card_id:
            raise ValueError("steal_card_id and give_card_id are required for Forced Deal.")

        steal_color_actual = find_card_color(target.properties, steal_card_id)
        give_color_actual = find_card_color(actor.properties, give_card_id)

        if not params.get("from_full_set_allowed", False):
            if is_full_set(target, steal_color_actual):
                raise ValueError("Cannot take from target full set.")
            if is_full_set(actor, give_color_actual):
                raise ValueError("Cannot give from your full set.")

        target.properties[steal_color_actual].remove(steal_card_id)
        actor.properties[give_color_actual].remove(give_card_id)

        actor.properties.setdefault(steal_color_actual, []).append(steal_card_id)
        target.properties.setdefault(give_color_actual, []).append(give_card_id)

        return {
            "type": "swap_property",
            "actor_id": actor_id,
            "target_id": target_player_id,
            "stolen_card_id": steal_card_id,
            "given_card_id": give_card_id,
            "stolen_color": steal_color_actual,
            "given_color": give_color_actual,
        }

    raise ValueError(f"Unsupported property manipulation effect: {effect}")


### Change Wild Color Handler function

In [ ]:
from typing import Dict

def change_wild_color(
    state: GameState,
    catalog: Dict[str, CardDef],
    player_id: str,
    card_id: str,
    new_color: str,
) -> GameState:
    if player_id not in state.players:
        raise ValueError(f"Unknown player_id: {player_id}")

    player = state.players[player_id]

    # Find current color pile containing this card
    current_color = None
    for color, cards in player.properties.items():
        if card_id in cards:
            current_color = color
            break
    if current_color is None:
        raise ValueError("Card not found in player properties.")

    if card_id not in catalog:
        raise ValueError(f"Unknown card id: {card_id}")
    cd = catalog[card_id]

    if cd.kind not in {"property_wild", "property"}:
        raise ValueError("Card is not a property or wild property.")
    if not cd.colors:
        raise ValueError("Card has no color options.")

    # Validate color choice
    if "any" in cd.colors:
        # optional: only allow real property colors
        if "RENT_TABLE" in globals() and new_color not in RENT_TABLE:
            raise ValueError(f"Invalid color '{new_color}'.")
    else:
        if new_color not in cd.colors:
            raise ValueError(f"Card cannot be set to color '{new_color}'.")

    # Move card to the new color pile
    player.properties[current_color].remove(card_id)
    player.properties.setdefault(new_color, []).append(card_id)

    return state

